# DAM405 Machine Learning Operations Practical 4
## Automated Training and Tuning with FLAML

**Module tutor:** Kamal Acharya

This practical uses FLAML, an open-source AutoML library, to search for a strong classification model within a fixed time budget. The AutoML result is compared with a simple logistic-regression baseline, inspected, and saved for later use.

## Learning Outcomes

By the end of this practical, I should be able to:

1. Use an open-source AutoML tool to train and tune a model.
2. Compare an AutoML model with a simple baseline model.
3. Inspect the best model, its hyperparameters, and its score.
4. Save the resulting model for later use.

## Background

Automated machine learning (AutoML) automates repetitive model-building decisions, including algorithm selection and hyperparameter tuning. It does not remove the need for human decisions: the practitioner still chooses the dataset, target, evaluation metric, time budget, validation strategy, and whether the result is suitable for the application.

## 1. Install and Import Dependencies

Run the following cell once for the active notebook kernel. The `%pip` syntax installs packages into the environment associated with this notebook.

In [1]:
%pip install -q "flaml[automl]" scikit-learn joblib pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
from pathlib import Path

import joblib
import pandas as pd
from flaml import AutoML
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TIME_BUDGET_SECONDS = 60

## 2. Load and Split the Dataset

The built-in breast-cancer dataset is a binary classification problem (569 samples, 30 features). An 80/20 train/test split is used. The test set is kept separate from model training and is used only for the final comparison.

**Deviations from the handout (intentional):**
- `stratify=y` keeps the benign/malignant class ratio the same in the training and test sets, which makes the comparison fairer on a small test set.
- `random_state` / `seed` are fixed for the baseline and FLAML so the experiment is as reproducible as possible. FLAML's search is time-bounded, so results can still vary slightly between machines and runs.

In [3]:
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Dataset shape: {X.shape}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

Dataset shape: (569, 30)
Training samples: 455
Test samples: 114


## 3. Build a Baseline Model

A baseline provides a reference point for judging whether AutoML adds value. Logistic regression is a transparent, strong starting point for this dataset. Its training time is also recorded so the compute cost can be compared with AutoML.

In [4]:
start = time.perf_counter()
baseline = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
baseline_seconds = time.perf_counter() - start

baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
baseline_roc_auc = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])

print(f"Baseline test accuracy: {baseline_accuracy:.4f}")
print(f"Baseline test ROC AUC : {baseline_roc_auc:.4f}")
print(f"Baseline training time: {baseline_seconds:.2f} s")

Baseline test accuracy: 0.9649
Baseline test ROC AUC : 0.9954
Baseline training time: 0.17 s


## 4. Run FLAML AutoML

FLAML searches across candidate classifiers (LightGBM, random forest, XGBoost, extra trees, SGD, L1 logistic regression, ...) and their hyperparameters. It optimises cross-validated accuracy on the training data only; the test set is never seen during the search. The search is limited to 60 seconds so that the computational cost is explicit.

In [5]:
automl = AutoML()
automl.fit(
    X_train,
    y_train,
    task="classification",
    metric="accuracy",
    time_budget=TIME_BUDGET_SECONDS,
    seed=RANDOM_STATE,
)

[flaml.automl.logger: 10-05 14:28:36] {2470} INFO - task = classification
[flaml.automl.logger: 10-05 14:28:36] {2481} INFO - Evaluation method: cv
[flaml.automl.logger: 10-05 14:28:36] {2584} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 10-05 14:28:36] {2701} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 10-05 14:28:36] {3006} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 10-05 14:28:36] {3141} INFO - Estimated sufficient time budget=538s. Estimated necessary time budget=12s.
[flaml.automl.logger: 10-05 14:28:36] {3192} INFO -  at 0.1s,	estimator lgbm's best error=9.4505e-02,	best estimator lgbm's best error=9.4505e-02
[flaml.automl.logger: 10-05 14:28:36] {3006} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 10-05 14:28:36] {3192} INFO -  at 0.1s,	estimator lgbm's best error=9.4505e-02,	best estimator lgbm's best error=9.4505e-02
[flaml.autom

## 5. Inspect the Best Model

The best estimator and configuration describe what FLAML selected. The cross-validation score is calculated as `1 - best_loss` because FLAML stores a loss (here `1 - accuracy`). It is the best validation score observed during the search and is different from the final test-set score below.

In [6]:
print("Best estimator:", automl.best_estimator)
print("Best configuration:")
for name, value in automl.best_config.items():
    print(f"  {name}: {value}")
print(f"Best cross-validation score: {1 - automl.best_loss:.4f}")

Best estimator: lgbm
Best configuration:
  n_estimators: 118
  num_leaves: 7
  min_child_samples: 11
  learning_rate: 0.5138753648083025
  log_max_bin: 5
  colsample_bytree: 0.9778726268416658
  reg_alpha: 0.014925762215576498
  reg_lambda: 0.4520553510696906
Best cross-validation score: 0.9780


## 6. Evaluate and Compare on the Test Set

Both models are evaluated on the same held-out test set. The difference is positive when AutoML performs better than the baseline, and is also reported in percentage points.

In [7]:
automl_accuracy = accuracy_score(y_test, automl.predict(X_test))
accuracy_difference = automl_accuracy - baseline_accuracy
wrong_baseline = round((1 - baseline_accuracy) * len(y_test))
wrong_automl = round((1 - automl_accuracy) * len(y_test))

print(f"AutoML test accuracy  : {automl_accuracy:.4f}  ({wrong_automl} errors / {len(y_test)})")
print(f"Baseline test accuracy: {baseline_accuracy:.4f}  ({wrong_baseline} errors / {len(y_test)})")
print(f"AutoML - baseline     : {accuracy_difference:+.4f} ({accuracy_difference * 100:+.2f} percentage points)")

AutoML test accuracy  : 0.9561  (5 errors / 114)
Baseline test accuracy: 0.9649  (4 errors / 114)
AutoML - baseline     : -0.0088 (-0.88 percentage points)


## 7. Save the Best Model

The complete FLAML object is saved so that the selected model can be loaded and used later without repeating the search. The file is reloaded once to confirm that it works.

In [8]:
model_path = Path("automl_model.joblib")
joblib.dump(automl, model_path)
print(f"Saved {model_path}")

reloaded = joblib.load(model_path)
reloaded_accuracy = accuracy_score(y_test, reloaded.predict(X_test))
print(f"Reloaded model test accuracy: {reloaded_accuracy:.4f}")

Saved automl_model.joblib
Reloaded model test accuracy: 0.9561


## 8. Optional Experiment: Time Budget and Metric

To study the cost/quality trade-off, the search is repeated with larger time budgets (120 s and 300 s) and once with `metric="roc_auc"`. ROC AUC measures how well the model ranks positive-class probabilities rather than how many labels are correct. For every run the best estimator, cross-validation score, test accuracy, test ROC AUC, and elapsed time are recorded. The models from these runs are only used for comparison; the saved model above is the 60-second accuracy run.

In [9]:
def run_automl(time_budget, metric):
    model = AutoML()
    start = time.perf_counter()
    model.fit(
        
        X_train,
        y_train,
        task="classification",
        metric=metric,
        time_budget=time_budget,
        seed=RANDOM_STATE,
        verbose=0,
    )
    elapsed = time.perf_counter() - start
    return {
        "metric": metric,
        "time_budget_s": time_budget,
        "elapsed_s": round(elapsed, 1),
        "best_estimator": model.best_estimator,
        "best_cv_score": round(1 - model.best_loss, 4),
        "test_accuracy": round(accuracy_score(y_test, model.predict(X_test)), 4),
        "test_roc_auc": round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 4),
    }


experiments = [
    {
        "metric": "baseline (LogReg)",
        "time_budget_s": None,
        "elapsed_s": round(baseline_seconds, 1),
        "best_estimator": "logistic_regression",
        "best_cv_score": None,
        "test_accuracy": round(baseline_accuracy, 4),
        "test_roc_auc": round(baseline_roc_auc, 4),
    },
    {
        "metric": "accuracy",
        "time_budget_s": TIME_BUDGET_SECONDS,
        "elapsed_s": None,
        "best_estimator": automl.best_estimator,
        "best_cv_score": round(1 - automl.best_loss, 4),
        "test_accuracy": round(automl_accuracy, 4),
        "test_roc_auc": round(roc_auc_score(y_test, automl.predict_proba(X_test)[:, 1]), 4),
    },
]
experiments.append(run_automl(120, "accuracy"))
experiments.append(run_automl(300, "accuracy"))
experiments.append(run_automl(60, "roc_auc"))

results = pd.DataFrame(experiments)
results

KeyboardInterrupt: 

Note: the elapsed time of the main 60-second run is not re-measured here (it is shown in the FLAML log above and is approximately equal to the budget). FLAML stops once the budget is used, so elapsed time is close to `time_budget_s` for every AutoML run.